# ROGII — Wellbore Geology Prediction
## Notebook 0: Competition Context & Domain Knowledge

> **Competition**: [Kaggle — ROGII Wellbore Geology Prediction](https://www.kaggle.com/competitions/rogii-wellbore-geology-prediction)  
> **Metric**: RMSE on True Vertical Thickness (TVT)  
> **Type**: Kernel-only competition (code runs on Kaggle servers at submission time)

## 1. Problem Overview

### What is Geosteering?

When drilling a horizontal oil/gas well, the drill bit must stay within a **thin productive reservoir layer** (often just a few meters thick). This process of steering the wellbore through the subsurface geological layers is called **geosteering**.

Traditional geosteering relies on:
- Real-time gamma-ray (GR) measurements from the drill bit
- Manual interpretation by geologists
- Delayed feedback loop (the geologist sees data *after* the bit has already drilled past)

**The problem**: human interpretation is slow, subjective, and cannot keep up with modern drilling speeds.

### ROGII's Solution

[ROGII](https://rogii.com/) builds geosteering software (StarSteer). This competition challenges ML practitioners to **automate the geological interpretation** step by predicting **True Vertical Thickness (TVT)** along horizontal wellbores.

## 2. What is True Vertical Thickness (TVT)?

TVT represents the **interpreted position** of the well within the geological layer stack — specifically, how far (in meters, measured vertically) the wellbore is from a reference geological boundary (typically the top of the reservoir formation).

```
Surface
  │
  │  (vertical section)
  │
  └──────────────────────────────── horizontal section
         │ GR log measured here │
         ↓                      ↓
    [shale above]          [shale below]
         │    [RESERVOIR LAYER]  │
         └──── TVT = distance from top of reservoir to the wellbore ────┘
```

A **lower RMSE on TVT** means the model can more accurately tell the driller:
*"You are X meters below the top of the reservoir — steer upward!"*

## 3. Data Description

The dataset contains **773 horizontal wells**, each with:

### Input Features (X)

| Column | Description | Unit |
|---|---|---|
| `DEPT` | Measured depth along the wellbore | meters |
| `GR` | Gamma-Ray log — distinguishes shale (high GR) from reservoir (low GR) | API units |
| `X`, `Y`, `Z` | 3D wellbore trajectory coordinates | meters |
| `AZIMUTH` | Azimuth angle of the wellbore | degrees |
| `INCLINATION` | Inclination angle from vertical | degrees |
| Reference well logs | GR and other logs from a paired vertical (type) well | various |

### Target (Y)

| Column | Description |
|---|---|
| `TVT` | True Vertical Thickness — position within the geological layer |

### Important Notes

- This is a **kernel-only competition**: at submission time, the 3-row test set shown on Kaggle is replaced by the real (large) test set
- **Validation strategy**: always split by well (not randomly) to prevent data leakage
- GR logs can drift between wells — **normalization per-well** is important

## 4. Gamma-Ray Log: The Core Signal

The **Gamma-Ray log** is the primary input signal. It measures natural radioactivity of the rock:

- **High GR (>75 API)**: Shale / clay — high radioactivity
- **Low GR (<50 API)**: Clean sandstone or carbonate — the reservoir
- **Medium GR**: Transitional lithology

```
GR ──→  HIGH |████████████| SHALE (above reservoir)
             |            |
        LOW  |   ░░░░░░   | RESERVOIR (target zone)
             |            |
        HIGH |████████████| SHALE (below reservoir)
```

The GR pattern along the horizontal wellbore encodes where the bit is within the layer stack. The ML model must learn to decode this signal into TVT.

## 5. Competition Leaderboard & Winning Strategies

Based on public notebooks and discussion:

| Approach | LB Score (RMSE) | Key Idea |
|---|---|---|
| Sample submission | ~15+ | Baseline |
| LightGBM (basic features) | ~12–15 | Tree boosting on raw logs |
| XGBoost Starter (cdeotte) | ~15 | Simple XGBoost |
| LGBM + feature engineering | ~10–12 | Rolling stats, lags |
| DWT-based + LGBM | **~9.25** | Wavelet decomposition of GR |
| Better solution (Tamrazov) | **~9.96** | Advanced feature engineering |
| Ensemble + particle filter | **<9** | Multi-model stacking + physics |

### Key Winning Techniques

1. **Validation by well**: 5-fold CV splitting by `well_id`, not rows
2. **~40 engineered features**: rolling means/std, lags, diff, ratios
3. **3D wellbore tortuosity**: most powerful domain-specific feature
4. **DWT on GR log**: Discrete Wavelet Transform extracts multi-scale frequency features
5. **GR normalization**: per-well or vs reference well normalization
6. **Model ensemble**: LightGBM + XGBoost + CatBoost + Neural Network
7. **Particle filter**: physics-informed post-processing for smooth TVT trajectory

## 6. Domain Literature

Relevant papers and resources:

- *Deep learning for prediction of complex geology ahead of drilling* ([arXiv:2104.02550](https://arxiv.org/abs/2104.02550))
- *AI-Driven Geosteering: Optimizing Horizontal Drilling Accuracy* — iFactory
- ROGII StarSteer documentation and use cases

### Physics-Based Intuition for Feature Engineering

```
GR signal along the wellbore ≈ "fingerprint" of the geology

Key physical relationships:
  - TVT changes smoothly (geology doesn't jump abruptly)
  - GR correlates with TVT but is noisy (borehole conditions)
  - The trajectory (X,Y,Z) constrains feasible TVT values
  - Reference well provides the "ground truth" GR pattern for this area
```

The best models combine:
- **Local features** (current GR value, lag-k values)
- **Global context** (well trajectory, comparison with reference well)
- **Frequency features** (DWT captures cyclic layer patterns)

## 7. Evaluation Metric: RMSE

$$\text{RMSE} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2}$$

- Units are in **meters** (same as TVT)
- A score of 9.25 means predictions are on average ~9.25 meters off
- Reservoir thickness is often 10–30m, so sub-5m accuracy would be excellent
- The metric is **symmetric** — over- and under-shooting are equally penalized
- Outliers dominate — smooth, consistent predictions matter more than occasional spikes

## 8. Strategy Roadmap

```
Phase 1 — Understand the data (EDA notebook)
  ├── Load and inspect train/test sets
  ├── Analyze GR distributions per well
  ├── Visualize TVT vs depth profiles
  ├── Understand trajectory data
  └── Identify missing values and outliers

Phase 2 — Feature Engineering (Modeling notebook)
  ├── Raw log features: GR, depth
  ├── Rolling statistics: mean, std, min, max (windows: 5, 10, 20, 50)
  ├── Lag features: GR at t-1, t-2, ..., t-k
  ├── Diff features: first and second derivative of GR
  ├── Trajectory features: tortuosity, dip angle, azimuth
  ├── DWT features: wavelet coefficients of GR
  └── Normalized GR vs reference well

Phase 3 — Modeling
  ├── Baseline: LightGBM
  ├── Advanced: XGBoost, CatBoost
  ├── Deep learning: LSTM / Transformer on sequence
  └── Ensemble: weighted blend / stacking

Phase 4 — Submission
  ├── Validate on 5-fold CV by well
  ├── Generate submission CSV
  └── Submit to Kaggle
```